In [10]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gymnasium as gym
from IPython.display import Image, display
import imageio

from collections import defaultdict
from tqdm import tqdm
from optuna.trial import FixedTrial

from src.logger import LogManager
from src.utils import load_policy, plot_learning_curve, plot_smoothed_learning_curve, q_stats
from src.optimization import param_opt_pipeline, get_params

In [11]:
env = gym.make("Acrobot-v1", render_mode = "rgb_array")
state = env.reset()

In [12]:
ANGLE_BINS = 10
VEL_BINS = 12

LOW = np.array([-np.pi, -np.pi, -6, -10])
HIGH = np.array([ np.pi,  np.pi,  6,  10])
BINS = np.array([ANGLE_BINS, ANGLE_BINS, VEL_BINS, VEL_BINS])

def transform_state(obs):
    cos1, sin1, cos2, sin2, w1, w2 = obs
    
    theta1 = np.arctan2(sin1, cos1)
    theta2 = np.arctan2(sin2, cos2)
    
    return np.array([theta1, theta2, w1, w2])

def discretise(obs):
    state = transform_state(obs)
    
    ratios = (state - LOW) / (HIGH - LOW)
    ratios = np.clip(ratios, 0, 1)
    
    discrete = (ratios * (BINS - 1)).astype(int)
    return tuple(discrete)

In [13]:
def select_action(policy, state, mode="greedy", epsilon=0.05, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    values = policy[state]
    n_actions = len(values)

    # Detect policy type by whether values sum to ~1 (probability dist) or not (Q-values)
    is_prob_policy = np.isclose(values.sum(), 1.0)

    if mode == "greedy":
        return int(np.argmax(values))

    elif mode == "stochastic":
        if is_prob_policy:
            return int(rng.choice(n_actions, p=values))
        else:
            # Softmax over Q-values as a fallback
            exp_q = np.exp(values - np.max(values))
            probs = exp_q / exp_q.sum()
            return int(rng.choice(n_actions, p=probs))

    elif mode == "epsilon_greedy":
        if rng.random() < epsilon:
            return int(rng.integers(n_actions))
        return int(np.argmax(values))

In [14]:
model = load_policy('SARSA')

In [15]:
model

defaultdict(<function src.utils.load_policy.<locals>.<lambda>()>,
            {(np.int64(4),
              np.int64(4),
              np.int64(5),
              np.int64(5)): array([-26.71817293, -26.2866315 , -25.70287549]),
             (np.int64(4),
              np.int64(4),
              np.int64(4),
              np.int64(6)): array([-21.96616083, -20.84179139, -19.89618632]),
             (np.int64(4),
              np.int64(4),
              np.int64(5),
              np.int64(6)): array([-24.00566982, -22.92268218, -22.4506878 ]),
             (np.int64(4),
              np.int64(4),
              np.int64(5),
              np.int64(4)): array([-22.30424425, -22.96713623, -23.91838024]),
             (np.int64(4),
              np.int64(5),
              np.int64(5),
              np.int64(5)): array([-22.87880334, -24.51239712, -24.61310999]),
             (np.int64(4),
              np.int64(4),
              np.int64(6),
              np.int64(5)): array([-20.39251973, -21.

In [16]:
evaluate_policy(model, mode="stochastic")

{'mean_reward': np.float64(-454.75),
 'std_reward': np.float64(66.82430321372607),
 'mean_steps': np.float64(455.15),
 'success_rate': np.float64(0.4)}

In [17]:
model.keys()

dict_keys([(np.int64(4), np.int64(4), np.int64(5), np.int64(5)), (np.int64(4), np.int64(4), np.int64(4), np.int64(6)), (np.int64(4), np.int64(4), np.int64(5), np.int64(6)), (np.int64(4), np.int64(4), np.int64(5), np.int64(4)), (np.int64(4), np.int64(5), np.int64(5), np.int64(5)), (np.int64(4), np.int64(4), np.int64(6), np.int64(5)), (np.int64(4), np.int64(4), np.int64(4), np.int64(5)), (np.int64(4), np.int64(3), np.int64(5), np.int64(5)), (np.int64(4), np.int64(4), np.int64(6), np.int64(4)), (np.int64(4), np.int64(3), np.int64(4), np.int64(6)), (np.int64(4), np.int64(5), np.int64(4), np.int64(6)), (np.int64(3), np.int64(5), np.int64(5), np.int64(5)), (np.int64(4), np.int64(5), np.int64(6), np.int64(4)), (np.int64(4), np.int64(3), np.int64(4), np.int64(5)), (np.int64(4), np.int64(5), np.int64(6), np.int64(5)), (np.int64(3), np.int64(4), np.int64(5), np.int64(5)), (np.int64(5), np.int64(3), np.int64(6), np.int64(4)), (np.int64(5), np.int64(3), np.int64(5), np.int64(5)), (np.int64(3), np.